# Session 1: Foundations & Your First AI Agent

**AI Agents Workshop Series — UCLA MQE Spring 2026**

---

### What You'll Build Today

By the end of this session, you will have a working AI agent that:

1. Calls the Claude API and receives structured, typed responses
2. Uses **tool calling** to query live economic data from FRED
3. Runs an autonomous **ReAct loop** — reasoning about what data to pull, pulling it, and synthesizing results

### Prerequisites

- Python 3.11+
- An Anthropic API key (set as `ANTHROPIC_API_KEY` environment variable)
- A FRED API key (free at https://fred.stlouisfed.org/docs/api/api_key.html)

### Notebook Structure

| Section | Topic | Time |
|---------|-------|------|
| §1 | Agents vs. Chatbots — the key distinction | 10 min |
| §2 | Your first API call + structured outputs | 15 min |
| §3 | Tool calling — letting Claude use functions | 20 min |
| §4 | The ReAct loop — building your FRED agent | 15 min |
| §5 | Exercises — extend and experiment | 15 min |

---

## §0 — Environment Setup

Run this cell once to install dependencies and verify your API keys.

In [ ]:
# Install required packages (uncomment if needed)
#!pip install anthropic fredapi pydantic tenacity pandas python-dotenv

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # loads .env file if present

# Verify API keys are set
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "Missing ANTHROPIC_API_KEY. Set it in your .env file or export it in your shell."
)
assert os.environ.get("FRED_API_KEY"), (
    "Missing FRED_API_KEY. Get one free at https://fred.stlouisfed.org/docs/api/api_key.html"
)

print("✓ All API keys loaded.")

---

## §1 — Agents vs. Chatbots: The Key Distinction

A **chatbot** is stateless and reactive:

```
User message → LLM → Response
```

An **agent** is goal-directed and autonomous. It follows the **ReAct** (Reason + Act) pattern:

```
Goal → Reason about what to do
     → Act (call a tool, fetch data)
     → Observe the result
     → Reason again (is the goal met?)
     → Repeat or return final answer
```

### Why this matters for quantitative economics

Consider the research question: *"Has the Phillips curve relationship strengthened since 2020?"*

A chatbot gives you a generic answer from its training data. An agent:

1. Decides it needs unemployment rate data (UNRATE) and inflation data (CPIAUCSL)
2. Pulls both series from FRED
3. Examines the correlation structure
4. Compares to pre-2020 patterns
5. Delivers a data-grounded analysis

The difference is architectural: the agent has **tools** and a **reasoning loop**. That's exactly what we'll build.

---

## §2 — Your First API Call + Structured Outputs

We'll start with the simplest possible interaction with Claude, then progressively add structure.

### 2.1 — Basic API Call

The Anthropic Python SDK provides a clean interface to Claude's Messages API. Every call requires three things: a **model**, a **max_tokens** limit, and a list of **messages**.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=512,
    messages=[
        {
            "role": "user",
            "content": "In two sentences, explain what the yield curve tells us about recession risk."
        }
    ]
)

print(response.content[0].text)

Let's inspect the response object to understand its structure. This is important — you'll use these fields throughout the workshop.

In [ ]:
# The response object has several useful fields
print(f"Model:        {response.model}")
print(f"Stop reason:  {response.stop_reason}")     # 'end_turn' = normal completion
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens:{response.usage.output_tokens}")
print(f"Content type: {response.content[0].type}")  # 'text' for normal responses

### 2.2 — The System Prompt

The **system prompt** shapes the model's behavior, persona, and output format. For quantitative work, we want Claude to be precise, cite data, and avoid hedging language.

Think of it as the "job description" you give the model before it starts working.

In [ ]:
SYSTEM_PROMPT = """You are a quantitative economics research assistant.

Rules:
- Be precise and data-driven. Cite specific numbers when available.
- If you don't have current data, say so explicitly rather than guessing.
- Use standard economic terminology appropriate for graduate-level readers.
- When presenting numbers, always include units and time periods.
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=512,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": "What is the current state of the U.S. labor market?"
        }
    ]
)

print(response.content[0].text)

Notice how the response is more precise and structured with the system prompt. But Claude is still working from its training data — it can't tell you *today's* unemployment rate. We'll fix that in §3.

### 2.3 — Structured Outputs with Pydantic

For programmatic use, we need the model to return **typed, validated data** — not free-form text. We use Pydantic models to define the expected schema, then instruct Claude to return JSON conforming to that schema.

This is a foundational pattern: every agent you build will use structured outputs to pass data between reasoning steps.

In [ ]:
from pydantic import BaseModel, Field


class MacroIndicator(BaseModel):
    """A single macroeconomic indicator with context."""
    name: str = Field(description="Indicator name, e.g. 'Real GDP Growth'")
    fred_series_id: str = Field(description="FRED series identifier, e.g. 'A191RL1Q225SBEA'")
    latest_value: float = Field(description="Most recent observed value")
    unit: str = Field(description="Unit of measurement, e.g. 'percent', 'index'")
    direction: str = Field(description="Recent trend: 'rising', 'falling', or 'stable'")
    significance: str = Field(description="One-sentence explanation of why this matters now")


class MacroSnapshot(BaseModel):
    """A structured snapshot of current U.S. macro conditions."""
    indicators: list[MacroIndicator]
    overall_assessment: str = Field(description="2-3 sentence synthesis of macro conditions")

In [ ]:
# Generate the JSON schema from the Pydantic model
schema = MacroSnapshot.model_json_schema()
print(json.dumps(schema, indent=2))

In [ ]:
# Ask Claude to return data conforming to our schema
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system=(
        "You are a macroeconomic data assistant. "
        "Respond ONLY with valid JSON conforming to the provided schema. "
        "No markdown, no commentary, no code fences — just the JSON object."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                f"Return the 4 most important U.S. macro indicators to watch this quarter.\n\n"
                f"Respond with JSON matching this schema:\n{json.dumps(schema, indent=2)}"
            )
        }
    ]
)

raw_text = response.content[0].text
print("Raw response (first 300 chars):")
print(raw_text[:300])

In [ ]:
# Parse and validate with Pydantic
snapshot = MacroSnapshot.model_validate_json(raw_text)

# Now we have typed, validated data
for indicator in snapshot.indicators:
    print(f"{indicator.name:30s}  {indicator.latest_value:>8.2f} {indicator.unit}")
    print(f"{'':30s}  FRED: {indicator.fred_series_id}  |  {indicator.direction}")
    print(f"{'':30s}  {indicator.significance}")
    print()

print("---")
print(snapshot.overall_assessment)

> **Key takeaway:** Structured outputs transform Claude from a text generator into a data source. Every downstream step — visualization, storage, comparison — becomes trivial when outputs are typed.
>
> **Critical problem:** Claude is returning values **from its training data**, which may be months or years stale. The numbers *look* authoritative (5.33%, 307.70, 4.10%) but are likely outdated. 
>
> ⚠️ A LLM's "memory" of numerical data should *never* be trusted in production. If a number matters for your analysis, it must come from a tool call to a live data source, not from the model's training data. We need live data. That's where tools come in.

---

## §3 — Tool Calling: Letting Claude Use Functions

**Tool calling** (also called "function calling") is the mechanism that turns a language model into an agent. Instead of answering from memory, the model can request to execute a function you've defined — then use the result to formulate its answer.

The flow:

```
User question
  → Claude reasons about what data it needs
  → Claude returns a tool_use block ("I want to call get_fred_series with series_id='UNRATE'")
  → Your code executes the function
  → You send the result back to Claude
  → Claude formulates a grounded answer
```

You define tools as JSON schemas. Claude decides **when** and **how** to call them.

### 3.1 — Connecting to the FRED API

First, let's set up our data function independently of Claude. Good practice: always test your tools in isolation before wiring them into an agent.

In [ ]:
import fredapi
import pandas as pd

fred = fredapi.Fred(api_key=os.environ["FRED_API_KEY"])


def get_fred_series(
    series_id: str,
    start_date: str = "2020-01-01",
    n_recent: int = 12
) -> str:
    """
    Fetch an economic time series from FRED.

    Parameters
    ----------
    series_id : str
        FRED series identifier (e.g., 'GDP', 'UNRATE', 'CPIAUCSL', 'DFF').
    start_date : str
        Observation start date in YYYY-MM-DD format.
    n_recent : int
        Number of most recent observations to return.

    Returns
    -------
    str
        JSON string with series metadata and recent observations.
    """
    try:
        data = fred.get_series(series_id, observation_start=start_date)
        info = fred.get_series_info(series_id)

        recent = data.dropna().tail(n_recent)

        result = {
            "series_id": series_id,
            "title": info["title"],
            "units": info["units"],
            "frequency": info["frequency"],
            "last_updated": str(info["last_updated"]),
            "observations": {
                str(date.date()): round(float(val), 4)
                for date, val in recent.items()
            }
        }
        return json.dumps(result, indent=2)

    except Exception as e:
        return json.dumps({"error": str(e), "series_id": series_id})

In [ ]:
# Test the function directly — always verify before giving it to an agent
result = get_fred_series("UNRATE", start_date="2024-01-01")
print(result)

### 3.2 — Defining the Tool Schema

Claude needs a JSON description of what the tool does, what parameters it accepts, and what it returns. This schema is how the model decides whether and how to use the tool.

Write tool descriptions the way you'd explain the function to a smart colleague who has never seen your codebase.

In [ ]:
tools = [
    {
        "name": "get_fred_series",
        "description": (
            "Fetch an economic time series from the Federal Reserve Economic Data (FRED) "
            "database. Returns recent observations with metadata. Use this tool whenever "
            "you need current or historical U.S. economic data — GDP, inflation (CPI/PCE), "
            "unemployment, interest rates, housing, employment, etc. "
            "Common series IDs: GDP, UNRATE, CPIAUCSL, PCEPI, DFF, T10Y2Y, PAYEMS, "
            "HOUST, RSAFS, UMCSENT."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "series_id": {
                    "type": "string",
                    "description": "FRED series ID (e.g., 'UNRATE' for unemployment rate)"
                },
                "start_date": {
                    "type": "string",
                    "description": "Start date in YYYY-MM-DD format. Default: '2020-01-01'"
                },
                "n_recent": {
                    "type": "integer",
                    "description": "Number of most recent observations to return. Default: 12"
                }
            },
            "required": ["series_id"]
        }
    }
]

### 3.3 — A Single Tool Call (Manual)

Before we build the full agent loop, let's walk through one tool-calling cycle manually. This makes the mechanism concrete.

In [ ]:
# Step 1: Send a question that requires live data
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    tools=tools,
    messages=[
        {"role": "user", "content": "What is the current U.S. unemployment rate?"}
    ]
)

print(f"Stop reason: {response.stop_reason}")
# 'tool_use' means Claude wants to call a function instead of responding directly

for block in response.content:
    print(f"\nBlock type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool name: {block.name}")
        print(f"  Arguments: {json.dumps(block.input, indent=2)}")
        print(f"  Tool ID:   {block.id}")
    elif block.type == "text":
        print(f"  Text: {block.text[:200]}")

In [ ]:
# Step 2: Execute the tool call and get the result
tool_block = next(b for b in response.content if b.type == "tool_use")
tool_result = get_fred_series(**tool_block.input)

print("Tool result:")
print(tool_result)

In [ ]:
# Step 3: Send the tool result back to Claude for synthesis
final_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    tools=tools,
    messages=[
        {"role": "user", "content": "What is the current U.S. unemployment rate?"},
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_block.id,
                    "content": tool_result
                }
            ]
        }
    ]
)

print(final_response.content[0].text)

> **What just happened:**
>
> 1. We asked a question requiring current data
> 2. Claude recognized it needed to call `get_fred_series` and told us which arguments to use
> 3. We executed the function and passed the result back
> 4. Claude used the **real data** to formulate an accurate, grounded answer
>
> This three-step handshake is the foundation of every AI agent. Now let's automate it.

---

## §4 — The ReAct Loop: Building Your FRED Agent

The manual process above works, but it's tedious. In practice, an agent needs to:

- Handle **multiple** tool calls in sequence (e.g., pull unemployment *then* inflation)
- Decide autonomously when it has enough information to answer
- Manage the full conversation history

We'll build this as a `while` loop that continues until Claude returns a final text response instead of a tool call.

### 4.1 — The Agent Function

In [ ]:
# Registry: maps tool names to Python functions
TOOL_REGISTRY = {
    "get_fred_series": get_fred_series,
}


def run_agent(
    user_query: str,
    tools: list[dict] = tools,
    system: str = SYSTEM_PROMPT,
    model: str = "claude-sonnet-4-20250514",
    max_iterations: int = 10,
    verbose: bool = True,
) -> str:
    """
    Run a ReAct agent loop.

    The agent will call tools as needed and return a final text answer
    once it has gathered enough information.

    Parameters
    ----------
    user_query : str
        The research question or task.
    tools : list[dict]
        Tool definitions (JSON schemas).
    system : str
        System prompt.
    model : str
        Claude model identifier.
    max_iterations : int
        Safety limit to prevent infinite loops.
    verbose : bool
        If True, print each step of the reasoning process.

    Returns
    -------
    str
        The agent's final response.
    """
    messages = [{"role": "user", "content": user_query}]
    total_input_tokens = 0
    total_output_tokens = 0

    for iteration in range(max_iterations):
        # Call Claude
        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=system,
            tools=tools,
            messages=messages,
        )

        # Track token usage
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens

        # Case 1: Claude is done — return the final answer
        if response.stop_reason == "end_turn":
            final_text = next(
                (b.text for b in response.content if b.type == "text"), ""
            )
            if verbose:
                print(f"\n{'='*60}")
                print(f"Agent complete in {iteration + 1} iteration(s)")
                print(f"Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")
                cost = (
                    total_input_tokens * 3.0 / 1_000_000
                    + total_output_tokens * 15.0 / 1_000_000
                )
                print(f"Estimated cost: ${cost:.4f}")
                print(f"{'='*60}\n")
            return final_text

        # Case 2: Claude wants to call a tool
        if response.stop_reason == "tool_use":
            # Add the assistant's response to history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  [{iteration+1}] Calling {block.name}({json.dumps(block.input)})")

                    # Look up and execute the function
                    func = TOOL_REGISTRY.get(block.name)
                    if func:
                        result = func(**block.input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {block.name}"})

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Send results back to Claude
            messages.append({"role": "user", "content": tool_results})

    return "Agent reached maximum iterations without completing."

### 4.2 — Test the Agent

Let's start with a simple single-tool query, then escalate to multi-tool reasoning.

In [ ]:
# Single data point
answer = run_agent("What is the latest U.S. unemployment rate and how does it compare to a year ago?")
print(answer)

In [ ]:
# Multi-tool reasoning: Claude should pull multiple series
answer = run_agent(
    "Compare CPI and PCE inflation measures since 2023. "
    "Which is running hotter and what might explain the divergence?"
)
print(answer)

In [ ]:
# Complex analytical question
answer = run_agent(
    "Assess current U.S. recession risk using the yield curve (T10Y2Y), "
    "unemployment rate, and consumer sentiment. "
    "What signal is each indicator sending?"
)
print(answer)

> **Observe the agent's behavior:**
>
> - It decides *which* FRED series to pull based on the question
> - It makes multiple tool calls when the question requires multiple data points
> - It synthesizes the results into a coherent analysis, citing the actual data
>
> You now have a functioning economic research agent. The rest of this notebook extends it.

### 4.3 — Adding a Second Tool

Real research agents need multiple data sources. Let's add a tool that returns metadata about a FRED series — useful when the agent needs to understand what a series measures before pulling data.

In [ ]:
def search_fred_series(search_text: str, limit: int = 5) -> str:
    """
    Search FRED for series matching a query.

    Useful when the agent knows what concept it wants (e.g., 'housing starts')
    but doesn't know the exact series ID.

    Parameters
    ----------
    search_text : str
        Free-text search query (e.g., 'core inflation', 'housing permits').
    limit : int
        Max number of results to return.

    Returns
    -------
    str
        JSON array of matching series with id, title, and frequency.
    """
    try:
        results = fred.search(search_text)
        top = results.head(limit)
        matches = []
        for sid, row in top.iterrows():
            matches.append({
                "series_id": sid,
                "title": row.get("title", ""),
                "frequency": row.get("frequency", ""),
                "units": row.get("units", ""),
                "popularity": int(row.get("popularity", 0)),
            })
        return json.dumps(matches, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


# Add to tool definitions
tools_extended = tools + [
    {
        "name": "search_fred_series",
        "description": (
            "Search the FRED database for economic series matching a query. "
            "Use this when you need to find the correct series ID for a concept. "
            "For example, search 'core inflation' to find CPILFESL, or "
            "'housing starts' to find HOUST."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "search_text": {
                    "type": "string",
                    "description": "Search query describing the economic concept"
                },
                "limit": {
                    "type": "integer",
                    "description": "Max results to return (default: 5)"
                }
            },
            "required": ["search_text"]
        }
    }
]

# Update the registry
TOOL_REGISTRY["search_fred_series"] = search_fred_series

In [ ]:
# Test with a question that requires searching first
answer = run_agent(
    "What's happening with U.S. housing starts? Pull the most recent data.",
    tools=tools_extended,
)
print(answer)

> **Notice the multi-step reasoning:** The agent may first *search* for the right series ID, then *fetch* the data. This search → retrieve pattern is fundamental to how agents solve open-ended problems.

---

## §5 — Exercises

Complete as many of these as time allows. They are ordered by difficulty.

### Exercise 1: Add a Calculation Tool (10 min)

Create a new tool called `compute_statistics` that takes a JSON string of data (like the output of `get_fred_series`) and returns basic statistics: mean, standard deviation, min, max, and the percent change from first to last observation.

**Steps:**
1. Write the Python function
2. Write the tool schema
3. Add both to `tools_extended` and `TOOL_REGISTRY`
4. Test: *"What is the average and volatility of the Fed Funds rate since 2022?"*

In [ ]:
# Exercise 1: Your code here
#
# def compute_statistics(data_json: str) -> str:
#     """Compute summary statistics from FRED series data."""
#     ...
#
# Hint: parse the JSON, extract the 'observations' dict,
#       convert values to a pandas Series, and compute stats.

### Exercise 2: Multi-Series Comparison (10 min)

Ask the agent a question that requires pulling 3+ series and comparing them. Observe how many iterations the loop takes and how the agent sequences its tool calls.

**Suggested prompts:**

- *"Compare GDP growth, industrial production, and retail sales since 2023. Which is the strongest?"*
- *"Pull the 10-year Treasury yield, 2-year Treasury yield, and Fed Funds rate. Is the yield curve inverted?"*
- *"What do employment, consumer sentiment, and housing starts tell us about the economy right now?"*

In [ ]:
# Exercise 2: Your code here
#
# answer = run_agent("your question here", tools=tools_extended)
# print(answer)

### Exercise 3: Structured Output Agent (15 min)

Combine §2 (structured outputs) with §4 (agent loop). Modify `run_agent` so that after the agent finishes its tool calls, it returns a structured `MacroSnapshot` object instead of free text.

**Approach:**
1. After the agent loop completes, make one more API call with the full conversation history
2. Use a system prompt that forces JSON output matching the `MacroSnapshot` schema
3. Parse the result with Pydantic

In [ ]:
# Exercise 3: Your code here
#
# def run_structured_agent(user_query: str) -> MacroSnapshot:
#     """Run the agent, then format the final answer as a structured MacroSnapshot."""
#     ...
#
# Hint: You can reuse run_agent() to gather data, then make a second
# API call that takes the gathered data and formats it as JSON.

### Exercise 4 (Challenge): Error Handling with Retries (10 min)

Real APIs fail. Add retry logic using the `tenacity` library so that:

- FRED API calls retry up to 3 times with exponential backoff
- If Pydantic validation fails on structured output, retry the LLM call (up to 2 attempts)
- The agent gracefully handles unknown series IDs

In [ ]:
# Exercise 4: Your code here
#
# from tenacity import retry, stop_after_attempt, wait_exponential
#
# @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, max=10))
# def get_fred_series_robust(series_id: str, ...) -> str:
#     ...